# 04 · Extensión de modelos no lineales sin fuga de información

Este notebook amplía la comparación del notebook 03 para predecir `risk_class_1h`: 0 = estable, 1 = riesgo de vaciado y 2 = riesgo de saturación. Prueba XGBoost si está disponible, una SVM lineal, una SVM con núcleo RBF y una red neuronal multicapa.

Conserva exactamente las particiones temporales ya creadas: `train`, `validation` y `test`. El test final permanece bloqueado hasta que se elija un modelo mediante validation.

## Dependencias y alcance computacional

Se necesita `scikit-learn`. XGBoost es opcional. Las SVM con núcleo RBF son costosas cuando hay muchas observaciones, por lo que se entrenan con una muestra estratificada menor; esta limitación queda documentada y no altera las particiones temporales ni el protocolo contra fuga.

In [ ]:
# Descomenta estas líneas solo si faltan librerías en tu kernel de Jupyter.
# %pip install scikit-learn
#%pip install xgboost

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC, SVC

# XGBoost se carga de forma opcional: el notebook continúa aunque no esté instalado.
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost no está instalado: se ejecutarán los modelos de scikit-learn.')

# Localizamos las carpetas del proyecto a partir de la ubicación del notebook.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
FEATURES_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
RESULTS_DIR = PROJECT_ROOT / 'Datos modelado' / 'resultados_modelos'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Por seguridad, el test final solo se evalúa cuando se cambie expresamente a True.
EVALUATE_FINAL_TEST = True

# El límite mantiene el notebook reproducible en equipos personales. Validation y test se cargan completos.
MAX_TRAIN_ROWS = 500_000
MAX_RBF_TRAIN_ROWS = 50_000
MAX_MLP_TRAIN_ROWS = 200_000
RANDOM_STATE = 42
CHUNK_SIZE = 100_000


## Variables, carga y particiones temporales

La etiqueta, las variables predictivas y `dataset_split` proceden del notebook 01. Se excluyen campos futuros y cualquier variable que defina directamente el riesgo. Solo se cargan filas con etiqueta no nula y con una de las tres particiones oficiales.

In [ ]:
TARGET = 'risk_class_1h'

NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']
READ_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET, 'dataset_split']

feature_files = sorted(FEATURES_DIR.glob('estacion_hora_features_*.csv'))
if not feature_files:
    raise FileNotFoundError(f'No se encontraron CSV de variables en {FEATURES_DIR}')

def count_train_rows() -> int:
    """Cuenta los casos etiquetados de train sin cargar todos los CSV en memoria."""
    total = 0
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=[TARGET, 'dataset_split'], chunksize=CHUNK_SIZE, low_memory=False):
            total += ((chunk['dataset_split'] == 'train') & chunk[TARGET].notna()).sum()
    return int(total)

train_total = count_train_rows()
train_fraction = min(1.0, MAX_TRAIN_ROWS / train_total)
print(f'Filas etiquetadas disponibles en train: {train_total:,}')
print(f'Fracción de train que se cargará: {train_fraction:.3%}')

def load_splits(train_fraction: float):
    """Carga train por muestra estratificada aproximada y validation/test completos."""
    rng = np.random.default_rng(RANDOM_STATE)
    parts = {'train': [], 'validation': [], 'test': []}

    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False):
            chunk = chunk[chunk[TARGET].notna() & chunk['dataset_split'].isin(parts)].copy()
            for split_name in parts:
                split_chunk = chunk[chunk['dataset_split'] == split_name]
                if split_name == 'train' and train_fraction < 1:
                    # Muestreamos dentro de cada clase para conservar aproximadamente su proporción.
                    keep = rng.random(len(split_chunk)) < train_fraction
                    split_chunk = split_chunk.loc[keep]
                if not split_chunk.empty:
                    parts[split_name].append(split_chunk)

    return tuple(pd.concat(parts[name], ignore_index=True) for name in ('train', 'validation', 'test'))

train, validation, test = load_splits(train_fraction)
for split_name, frame in [('train', train), ('validation', validation), ('test', test)]:
    print(f'{split_name}: {len(frame):,} filas | distribución: {frame[TARGET].value_counts(normalize=True).sort_index().round(4).to_dict()}')


## Preprocesamiento sin fuga de información

Los valores faltantes se imputan y las categorías se codifican usando solo `X_train`. Validation y test se transforman con los objetos ya aprendidos. La posterior reducción SVD y la estandarización de SVM RBF/red neuronal también se ajustan solo con train.

In [ ]:
X_train = train[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = train[TARGET].astype(int)
X_validation = validation[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_validation = validation[TARGET].astype(int)
X_test = test[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test = test[TARGET].astype(int)

numeric_pipeline = Pipeline(steps=[
    # La mediana se calcula únicamente con train cuando se llama a fit_transform.
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Las categorías no vistas después en validation/test no generan una columna nueva.
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
])

# Ajuste permitido: solo datos de entrenamiento.
X_train_ready = preprocessor.fit_transform(X_train)
# Transformaciones sin ajuste: validation y test no contribuyen a ningún parámetro aprendido.
X_validation_ready = preprocessor.transform(X_validation)
X_test_ready = preprocessor.transform(X_test)

print('Matriz train:', X_train_ready.shape)
print('Matriz validation:', X_validation_ready.shape)


## Funciones de apoyo y muestras para modelos costosos

La SVM lineal y XGBoost pueden usar la muestra general de train. La SVM RBF necesita una muestra menor debido a su coste cuadrático aproximado. La red neuronal usa una muestra intermedia. En los tres casos las observaciones proceden exclusivamente de train.

In [ ]:
def stratified_indices(target: pd.Series, maximum_rows: int, random_state: int) -> np.ndarray:
    """Devuelve índices de una muestra estratificada, siempre tomada de train."""
    if len(target) <= maximum_rows:
        return np.arange(len(target))

    rng = np.random.default_rng(random_state)
    selected = []
    for class_value, class_positions in target.groupby(target).groups.items():
        positions = np.asarray(list(class_positions))
        class_size = max(1, round(maximum_rows * len(positions) / len(target)))
        selected.append(rng.choice(positions, size=min(class_size, len(positions)), replace=False))
    return np.sort(np.concatenate(selected))

def evaluate(model, features, target, name: str) -> dict:
    """Calcula métricas robustas cuando las tres clases no tienen la misma frecuencia."""
    prediction = model.predict(features)
    print(f'\n--- {name} ---')
    print('Balanced accuracy:', round(balanced_accuracy_score(target, prediction), 4))
    print('F1 macro:', round(f1_score(target, prediction, average='macro'), 4))
    print('Matriz de confusión (filas: real; columnas: predicción):')
    print(confusion_matrix(target, prediction, labels=[0, 1, 2]))
    print(classification_report(target, prediction, labels=[0, 1, 2], target_names=['estable', 'vaciado', 'saturación'], zero_division=0))
    return {
        'model': name,
        'balanced_accuracy': balanced_accuracy_score(target, prediction),
        'f1_macro': f1_score(target, prediction, average='macro'),
    }

rbf_indices = stratified_indices(y_train, MAX_RBF_TRAIN_ROWS, RANDOM_STATE)
mlp_indices = stratified_indices(y_train, MAX_MLP_TRAIN_ROWS, RANDOM_STATE + 1)
print(f'Muestra train para SVM RBF: {len(rbf_indices):,} filas')
print(f'Muestra train para red neuronal: {len(mlp_indices):,} filas')


## Entrenamiento y comparación en validation

La SVM lineal usa directamente la matriz dispersa. Para la SVM RBF y la red neuronal, `TruncatedSVD` reduce la matriz codificada y `StandardScaler` estandariza sus componentes. Ambos ajustes se hacen sobre train y se aplican sin reajuste a validation.

In [ ]:
fitted_models = {}
results = []

# La SVM lineal es una referencia de margen máximo eficiente con variables codificadas dispersas.
linear_svm = LinearSVC(class_weight='balanced', C=1.0, max_iter=5_000, random_state=RANDOM_STATE)
linear_svm.fit(X_train_ready, y_train)
fitted_models['SVM lineal'] = linear_svm
results.append(evaluate(linear_svm, X_validation_ready, y_validation, 'SVM lineal · validation'))

# XGBoost modela interacciones no lineales; usa el método hist para controlar tiempo y memoria.
if XGBOOST_AVAILABLE:
    xgboost = XGBClassifier(
        objective='multi:softprob', num_class=3, eval_metric='mlogloss',
        n_estimators=350, max_depth=8, learning_rate=0.08,
        subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE,
    )
    xgboost.fit(X_train_ready, y_train)
    fitted_models['XGBoost'] = xgboost
    results.append(evaluate(xgboost, X_validation_ready, y_validation, 'XGBoost · validation'))

# Reducimos y escalamos solo con train antes de aplicar modelos sensibles a la escala y dimensionalidad.
n_components = min(150, X_train_ready.shape[1] - 1)
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
X_train_reduced = svd.fit_transform(X_train_ready)
X_validation_reduced = svd.transform(X_validation_ready)
X_test_reduced = svd.transform(X_test_ready)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_reduced)
X_validation_scaled = scaler.transform(X_validation_reduced)
X_test_scaled = scaler.transform(X_test_reduced)

# La SVM RBF se entrena en una muestra de train por su elevado coste con grandes volúmenes.
rbf_svm = SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced', cache_size=2_000, random_state=RANDOM_STATE)
rbf_svm.fit(X_train_scaled[rbf_indices], y_train.iloc[rbf_indices])
fitted_models['SVM RBF'] = rbf_svm
results.append(evaluate(rbf_svm, X_validation_scaled, y_validation, 'SVM RBF · validation'))

# La red neuronal multicapa usa parada temprana y una muestra de train para limitar el sobreajuste y el tiempo.
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
    alpha=1e-4, batch_size=512, learning_rate_init=1e-3,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=12,
    max_iter=150, random_state=RANDOM_STATE,
)
mlp.fit(X_train_scaled[mlp_indices], y_train.iloc[mlp_indices])
fitted_models['Red neuronal MLP'] = mlp
results.append(evaluate(mlp, X_validation_scaled, y_validation, 'Red neuronal MLP · validation'))

results_validation = pd.DataFrame(results).sort_values(['f1_macro', 'balanced_accuracy'], ascending=False).reset_index(drop=True)
results_validation.to_csv(RESULTS_DIR / 'comparacion_extension_no_lineales_validation.csv', index=False, encoding='utf-8-sig')
results_validation


In [ ]:
results_validation['model'] = (
    results_validation['model']
    .str.replace(' · validation', '', regex=False)
)

results_validation

## Lectura del resultado y test final

Selecciona el modelo por sus métricas en validation. Solo después, cambia `EVALUATE_FINAL_TEST` a `True` y ejecuta esta celda una única vez. El modelo ganador usa las transformaciones ya ajustadas exclusivamente con train.

In [ ]:
best_name = results_validation.iloc[0]['model']
best_model = fitted_models[best_name]

# Cada modelo necesita su representación correspondiente, creada sin usar validation/test para ajustar parámetros.
if best_name in {'SVM RBF', 'Red neuronal MLP'}:
    final_features = X_test_scaled
else:
    final_features = X_test_ready

if EVALUATE_FINAL_TEST:
    final_result = evaluate(best_model, final_features, y_test, f'Test final: {best_name}')
    pd.DataFrame([final_result]).to_csv(RESULTS_DIR / 'extension_no_lineales_test_final.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame([final_result])
else:
    print(f'Mejor modelo provisional en validation: {best_name}')
    print('Test bloqueado. Cambia EVALUATE_FINAL_TEST a True solo tras cerrar la selección con validation.')


## Interpretación para el TFM

Compara este archivo de resultados con `comparacion_no_lineales_validation.csv` generado por el notebook 03 y con la línea base del notebook 02. La decisión debe priorizar F1 macro y balanced accuracy, además de considerar el coste de entrenamiento, la estabilidad y la interpretabilidad del modelo.